## Preprocesamiento y Modelado

Una vez inspeccionado el dataset en `customer_churn_eda.ipynb` definimos una una estrategia de preprocesamiento iterativo (de menos a más) para el encontrar

In [60]:
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.preprocessing import OneHotEncoder,RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, f1_score, make_scorer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.discriminant_analysis import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from feature_engine.imputation import RandomSampleImputer
from xgboost import XGBClassifier


### Configuración de constantes, rutas y variables 

En esta sección definimos constantes, rutas de archivos y atributos del dataset

In [61]:
# Rutas de los archivos de datos
TRAIN_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/train.csv"
TEST_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Cargamos los datos
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
# Definir variables objetivo
TARGET = 'Exited'
# Variables numéricas: Incluyo las continuas y las binarias numéricas (HasCrCard, IsActiveMember)
NUM_FEATURES = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
# Variables categóricas: Las de texto con pocas categorías
CAT_FEATURES = ['Geography', 'Gender', 'Surname']
# Variables a eliminar inicialmente (IDs y apellido)
DROP_FEATURES = ['CustomerId']
SURNAME_COL = 'Surname'
RANDOM_STATE = 100            # Semilla para reproducibilidad

### 1. Preparación de Datos

Separamos variables independientes y dependientes en X_train e y_train por convención.

In [62]:
# X_train = variables independientes
# y_train = variable dependiente u objetivo
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

## Clase para añadir features 

Creamos una clase para crear features sin fuga de datos. Esta clase aprende estadísticos (medianas y frecuencias) durante el fold de entrenamiento (`fit()`) y crea columnas nuevas usando esos estadístico, sin mirar la variable objetivo y sin fuga de datos tranformación(`tranform()`). No imputa ni escala de forma definitiva.

Activamos por familias.



In [63]:
# ---------- Feature Engineering ----------
from sklearn.base import BaseEstimator, TransformerMixin
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Genera features nuevas de forma segura (sin fuga):
    aprende mediana/frecuencias en fit() y las usa en transform().
    Activación por familias para ir de menos a más.
    """
    def __init__(
        self,
        # cada flag activa un grupo de features nuevas para probar mejoras
        # de forma incremental
        add_missing_count=True,     # cuenta de missings
        add_balance=True,           # añade features de Balance/Salary/ratios
    ):
        self.add_missing_count = add_missing_count
        self.add_balance = add_balance
   

    def fit(self, X, y=None):
        X = X.copy()
        # Calculamos medianas de columnas clave
        # lo hacemos en fit porque en validación cruzada, cada fold tiene un “train interno”
        # Si calculamos las medianas con todo el dataset,estaríamos usando info del fold de validación 
        # (fuga de datos) hacia el train
        # La hacemos en fit() para garantizar que cada fold de CV aprende sus propias medianas
        # Usamos medianas para evitar outliers y crear features sin NaN
        self.balance_median_ = X["Balance"].median(skipna=True)
        self.salary_median_ = X["EstimatedSalary"].median(skipna=True)
        self.creditscore_median_ = X["CreditScore"].median(skipna=True)
        self.age_median_ = X["Age"].median(skipna=True)
        # para la variable NumOfProducts (discreta) usamos moda (valor más frecuente)
        self.numprod_mode_ = X["NumOfProducts"].mode(dropna=True).iloc[0]

        return self

    def transform(self, X):
        # Copia para retornarlo con las nuevas features añadidas
        X = X.copy()
        
        # --- Missing count por fila ---
        # add_indicator ya añade columnas binarias (crea una feature) por cada columna con missings
        # pero aquí añadimos una feature con el conteo total de missings por fila
        # La falta de datos indica el perfil del cliente,
        # un cliente con muchos datos faltantes puede ser menos comprometido, menos activo, menos interesado, etc.
        if self.add_missing_count:
            # columnas a considerar para el conteo de missings
            miss_cols = ["CreditScore","Balance","NumOfProducts","EstimatedSalary","HasCrCard"]
            # solo las que existan
            miss_cols = [c for c in miss_cols if c in X.columns]
            X["missing_count"] = X[miss_cols].isna().sum(axis=1)

        # --- Balance / Salary / ratios ---
        # features basadas en Balance y EstimatedSalary
        # Creamos "señales" de tipo "tengo saldo o no", "saldo cero", etc.
        # Creamos features logarítmicas para reducir el impacto de outliers y colas largas
        # los logaritmos ayudan a modelos lineales a capturar relaciones no lineales
        # La relación entre Balance y Salary puede indicar el nivel de ahorro o gasto del cliente
        # Creamos ratios entre Balance y Salary y su logaritmo para capturar la relación entre ambos
        # Si un cliente tiene un balance alto en comparación con su salario, puede indicar una mayor estabilidad financiera
        # o capacidad de ahorro, lo cual puede influir en su probabilidad de abandono
        if self.add_balance:
            balance_raw = X["Balance"]
            balance = balance_raw.fillna(self.balance_median_)
            salary = X["EstimatedSalary"].fillna(self.salary_median_)
            # features binarias
            X["HasBalance"] = (balance_raw > 0).fillna(False).astype(int)     # 1 si Balance > 0 (NaN -> 0)
            X["Balance_is_zero"] = (balance_raw == 0).fillna(False).astype(int) # 1 si Balance == 0 (NaN -> 0).
            # features logarítmicas
            X["LogBalance"] = np.log1p(np.clip(balance, 0, None))
            X["LogSalary"] = np.log1p(np.clip(salary, 0, None))
            # ratios Balance/Salary
            X["BalToSal"] = balance / (salary + 1.0) # evitar división por cero
            X["LogBalToSal"] = np.log1p(np.clip(X["BalToSal"], 0, None)) # log(1 + ratio)

        return X  # retornamos el DataFrame con las nuevas features añadidas

---------------------------------------

### Funciones auxiliar
#### Construir preprocesadores básico

La siguiente función genera un preprocesador básico que constituye la línea base de la que partimos en versiones anteriores.

In [64]:

def make_basic_preprocessor(X: pd.DataFrame):
    """Genera un ColumnTransformer con pipelines de preprocesamiento
    para variables numéricas y categóricas según la versión indicada.

    Args:
        X (pd.DataFrame): _input data frame_
        version (str): Versión del preprocesamiento en formato 'N#_C#'
        e.g. 'N3_C1' donde N# indica la versión numérica y C# la categórica
        numerical_cols (_type_): columnas numéricas para el preprocesamiento
        categorical_cols (_type_): columnas categóricas para el preprocesamiento

    Raises:
        ValueError: _unknown numeric version_
        ValueError: _unknown categorical version_

    Returns:
        _type_: ColumnTransformer con pipelines de preprocesamiento
    """
    # Base Line 
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con mediana + indicador de faltantes + escalado estandar
    numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
    categorical_cols = ['Geography', 'Gender']
    num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ])
    # --- Categorical pipeline (si aplica) ---
    # Categóricas normales: imputar + onehot
    categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
   
    # Construimos el ColumnTransformer final
    # que une pipelines numéricos + pipelines categóricos
    col_trans_preprocessor = ColumnTransformer(
        transformers=[("num", num_pipe, numerical_cols), 
                      ("cat", categorical_pipe, categorical_cols)], 
        remainder="drop", # elimina columnas no especificadas
        verbose_feature_names_out=True) # nombres detallados de columnas
    return col_trans_preprocessor


#### Construir el preprocesador

In [ ]:
# ---------- Preprocessor builder ----------
from sklearn.preprocessing import PowerTransformer


def make_preprocessor():
    # Columnas base
    numerical_continuous_cols = ["CreditScore", "Age", "Balance", "EstimatedSalary"]
    numerical_discrete_cols = ["Tenure", "NumOfProducts"]
    numerical_binary_cols = ["HasCrCard", "IsActiveMember"]

    # --- Columnas features por grupo ---
    continuous_fe = []  # continuas creadas 
    discrete_fe = []  # discretas creadas
    binary_fe  = []  # binarias creadas

    # features que pueden existir según flags de FeatureEngineer
    discrete_fe += ["missing_count"]
    continuous_fe += ["LogBalance", "LogSalary", "BalToSal", "LogBalToSal"]
    binary_fe  += ["HasBalance", "Balance_is_zero"]

    # --- Pipelines numéricos ---
    continuous_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("power", PowerTransformer(method="yeo-johnson")),
        ("scaler", RobustScaler()),
    ])

    # Feature Eng continuas: mejor NO aplicar power otra vez 
    continuous_fe_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", RobustScaler()), # RobustScaler para features con outliers
    ])

    discrete_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent", add_indicator=True)),
    ])

    binary_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent", add_indicator=True)),
    ])

    # --- Categóricas ---
    cat_cols = ["Geography", "Gender"]
    
    # ONE HOT ENCODER para convertir categóricas en numericas
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
    )
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", one_hot_encoder),
    ])

    transformers = [
        ("cont", continuous_pipe, numerical_continuous_cols),
        ("cont_eng", continuous_fe_pipe, continuous_fe),
        ("disc", discrete_pipe, numerical_discrete_cols + discrete_fe),
        ("bin", binary_pipe, numerical_binary_cols + binary_fe),
        ("cat", categorical_pipe, cat_cols),
    ]

    preprocesor = ColumnTransformer(
        transformers=transformers,  # transformers 
        remainder="drop",           # elimina columnas no especificadas
        verbose_feature_names_out=True, # nombres detallados de columnas
        sparse_threshold=0.0 
    )

    return preprocesor

#### Construir pipelines

In [ ]:
# Creación y evaluación del pipeline
# Construcción del pipeline con feature engineer, preprocesador y modelo

def make_pipeline(feature_engineer:FeatureEngineer, preprocessor: ColumnTransformer, model=None):
    """Construye un Pipeline con el preprocesador y el modelo indicado.
    Args:
        feature_engineer (FeatureEngineer): Objeto FeatureEngineer para crear nuevas features
        preprocessor (ColumnTransformer): Preprocesador ColumnTransformer
        model (_type_, optional): Modelo de clasificación. Defaults to None.
    Returns:
        Pipeline: Pipeline con preprocesador y modelo, si no se indica modelo
        se usa LinearDiscriminantAnalysis por defecto.
    """
    if model is None:
        model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
        #model = LogisticRegression(max_iter=10000,solver="saga", random_state=RANDOM_STATE,class_weight='balanced',n_jobs=-1)
    if feature_engineer is None:
        return Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", model),
        ])
    else:
        return Pipeline([
        ("feature_engineer", feature_engineer),
        ("preprocessor", preprocessor),
        ("classifier", model),
        ])




#### Evaluar pipelines

Esta función evalua el pipeline usando validación cruzada estratificada

In [67]:
def evaluate_pipeline(pipe: Pipeline, X: pd.DataFrame, y: pd.Series, n_splits=5):
    """ Evalúa el pipeline usando Validación Cruzada estratificada
    Args:
        pipe (Pipeline): Pipeline a evaluar.
        X (pd.DataFrame): Datos de entrada.
        y (pd.Series): Datos objetivo.
        n_splits (int, optional): número de divisiones para Stratified K-Fold. Por defecto es 5.
    Returns:
        dict: Diccionario con las métricas promedio y desviación estándar.
    """
    # Configuramos la Validación Local Cruzada  n_splits splits (divisiones)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    # Definimos las métricas que queremos extraer
    # f1, roc_auc, precision, recall, accuracy son strings estándar de sklearn.
    # Kappa requiere make_scorer.
    scoring_metrics = {
        "f1": "f1",
        "roc_auc": "roc_auc",
        "accuracy": "accuracy",
        'kappa': make_scorer(cohen_kappa_score),
        'precision': 'precision',
        'recall': 'recall',
    }
    cv_results = cross_validate(pipe, X, y, cv=cv, scoring=scoring_metrics, n_jobs=-1)
    return {
        "f1_mean": cv_results["test_f1"].mean(),
        "f1_std":  cv_results["test_f1"].std(),
        "auc_mean": cv_results["test_roc_auc"].mean(),
        "auc_std":  cv_results["test_roc_auc"].std(),
        "accuracy_mean": cv_results["test_accuracy"].mean(),
        "accuracy_std":  cv_results["test_accuracy"].std(),
        "kappa_mean": cv_results["test_kappa"].mean(),
        "kappa_std":  cv_results["test_kappa"].std(),
        "precision_mean": cv_results["test_precision"].mean(),
        "precision_std":  cv_results["test_precision"].std(),
        "recall_mean": cv_results["test_recall"].mean(),
        "recall_std":  cv_results["test_recall"].std()
    }

------------------
## Evaluación de diferentes features engineer y modelos

Probamos las diferentes variables que vamos creando en la clase `FeatureEngineer`. 

In [68]:
# Configuración de experimentos
# modificamos Feature Engineer a probar 
# Feature Engineer 
feature_engineer = FeatureEngineer(
    add_missing_count=True,
    add_balance=True
)


# Modelos a probar
models = {
    # Modelos lineales
    "LinearDiscriminantAnalysis": LinearDiscriminantAnalysis(),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"),
    # Arboles
    # No necesitan escalado de variables
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=400,random_state=RANDOM_STATE, 
                                           class_weight="balanced",n_jobs=-1,),
    "RandomForest_bal_subsample": RandomForestClassifier(n_estimators=400,random_state=RANDOM_STATE, 
                                           class_weight="balanced_subsample",n_jobs=-1,),
    # Otros modelos
    "NaiveBayes": GaussianNB(),
    "RedesNeurales": MLPClassifier(hidden_layer_sizes=(50,30), max_iter=1000, random_state=RANDOM_STATE,
                                   activation='relu',solver='adam',early_stopping=True),
    "KNN_5": KNeighborsClassifier(n_neighbors=5, n_jobs=-1,),
    "KNN_5_distance": KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1),


    
}



#### Preprocesador base



In [69]:
preprocesors_results = []
base_preprocesor = make_basic_preprocessor(train_df)
# pipeline sin feature engineer y modelo por defecto
pipe = make_pipeline(None,base_preprocesor,None) 
print("Evaluando preprocesador base")
cv_metrics = evaluate_pipeline(pipe, X_train, y_train , n_splits=5)
preprocesors_results.append({"version": "Base", **cv_metrics})
display(pd.DataFrame(preprocesors_results))

Evaluando preprocesador base


,version,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
0,Base,0.32825,0.017747,0.768164,0.008146,0.80725,0.005585,0.238353,0.018927,0.567959,0.036011,0.231288,0.01535


#### Añadimos feature engineer

Creamos nuevas variables con la clase `FeatureEngineer`

In [70]:
preprocesor = make_preprocessor()
pipe = make_pipeline(feature_engineer, preprocesor)
cv_metrics = evaluate_pipeline(pipe, X_train, y_train , n_splits=5)
preprocesors_results.append({"version": "v1", **cv_metrics})

results_df = pd.DataFrame(preprocesors_results).sort_values("f1_mean", ascending=False)
display(results_df)
best_preprocesor_version = results_df.iloc[0]["version"]
print("Mejor versión de preprocesador:", best_preprocesor_version)

{'force_int_remainder_cols': 'deprecated', 'n_jobs': None, 'remainder': 'drop', 'sparse_threshold': 0.0, 'transformer_weights': None, 'transformers': [('cont', Pipeline(steps=[('imputer',
                 SimpleImputer(add_indicator=True, strategy='median')),
                ('power', PowerTransformer()), ('scaler', RobustScaler())]), ['CreditScore', 'Age', 'Balance', 'EstimatedSalary']), ('cont_eng', Pipeline(steps=[('imputer',
                 SimpleImputer(add_indicator=True, strategy='median')),
                ('scaler', RobustScaler())]), ['LogBalance', 'LogSalary', 'BalToSal', 'LogBalToSal']), ('disc', Pipeline(steps=[('imputer',
                 SimpleImputer(add_indicator=True, strategy='most_frequent'))]), ['Tenure', 'NumOfProducts', 'missing_count']), ('bin', Pipeline(steps=[('imputer',
                 SimpleImputer(add_indicator=True, strategy='most_frequent'))]), ['HasCrCard', 'IsActiveMember', 'HasBalance', 'Balance_is_zero']), ('cat', Pipeline(steps=[('imputer', SimpleI

,version,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
1,v1,0.332573,0.022661,0.771169,0.009043,0.81375,0.003331,0.250196,0.020729,0.616078,0.022045,0.228221,0.020586
0,Base,0.328250,0.017747,0.768164,0.008146,0.80725,0.005585,0.238353,0.018927,0.567959,0.036011,0.231288,0.015350


Mejor versión de preprocesador: v1


#### Búsqueda del mejor modelo

Evaluamos los modelos con el mejor preprocesador encontrado 

In [71]:
model_results = []
# evaluamos todos los modelos con el preprocesador
for model_name, model in models.items():
     # Construimos el preprocesador fijo con la mejor versión
    preprocesor = make_preprocessor()
    pipe = make_pipeline(feature_engineer, preprocesor, model) 
    cv_metrics = evaluate_pipeline(pipe, X_train, y_train , n_splits=5)
    model_results.append({"modelo": model_name, **cv_metrics})

results_df = pd.DataFrame(model_results).sort_values("f1_mean", ascending=False)
best_model_name = results_df.iloc[0]["modelo"]
print("---- Resultados de validación cruzada de modelos con preprocesador fijo:----")
display(results_df)
print("Mejor modelo:", best_model_name)



{'force_int_remainder_cols': 'deprecated', 'n_jobs': None, 'remainder': 'drop', 'sparse_threshold': 0.0, 'transformer_weights': None, 'transformers': [('cont', Pipeline(steps=[('imputer',
                 SimpleImputer(add_indicator=True, strategy='median')),
                ('power', PowerTransformer()), ('scaler', RobustScaler())]), ['CreditScore', 'Age', 'Balance', 'EstimatedSalary']), ('cont_eng', Pipeline(steps=[('imputer',
                 SimpleImputer(add_indicator=True, strategy='median')),
                ('scaler', RobustScaler())]), ['LogBalance', 'LogSalary', 'BalToSal', 'LogBalToSal']), ('disc', Pipeline(steps=[('imputer',
                 SimpleImputer(add_indicator=True, strategy='most_frequent'))]), ['Tenure', 'NumOfProducts', 'missing_count']), ('bin', Pipeline(steps=[('imputer',
                 SimpleImputer(add_indicator=True, strategy='most_frequent'))]), ['HasCrCard', 'IsActiveMember', 'HasBalance', 'Balance_is_zero']), ('cat', Pipeline(steps=[('imputer', SimpleI

,modelo,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
4,RandomForest_bal_subsample,0.505744,0.013637,0.830720,0.009187,0.845375,0.003804,0.423882,0.015247,0.725095,0.017225,0.388344,0.012053
3,RandomForest,0.502153,0.014214,0.830914,0.008457,0.844625,0.004043,0.420130,0.015953,0.723258,0.019182,0.384663,0.012513
1,LogisticRegression,0.493700,0.010309,0.771588,0.009268,0.703250,0.003522,0.310387,0.012568,0.378353,0.005852,0.710429,0.023570
2,DecisionTree,0.477300,0.016490,0.672240,0.010222,0.785250,0.009317,0.342230,0.022231,0.474082,0.022087,0.480982,0.015927
6,RedesNeurales,0.379750,0.028553,0.778555,0.008908,0.820375,0.008354,0.294553,0.032302,0.640991,0.047948,0.269939,0.021252
8,KNN_5_distance,0.329160,0.025305,0.702127,0.011059,0.799625,0.004569,0.229394,0.025075,0.516862,0.024326,0.241718,0.022407
0,LinearDiscriminantAnalysis,0.328556,0.021520,0.772294,0.009229,0.811875,0.002437,0.244670,0.018937,0.601915,0.013444,0.226380,0.019917
7,KNN_5,0.327157,0.021167,0.696223,0.011884,0.802875,0.002921,0.231872,0.019970,0.536284,0.014813,0.235583,0.018949
5,NaiveBayes,0.166091,0.106624,0.720199,0.014508,0.795500,0.002750,0.108677,0.075461,0.505626,0.027505,0.113497,0.099889


Mejor modelo: RandomForest_bal_subsample


## Construcción del pipeline final para Kaggle

Una vez obtenido la mejor combinación de preprocesadores y el mejor modelo, construimos el pipeline final para kaggle con la mejor combinación de ambos y volvemos a ejecutar la evaluación y el entrenamiento para finalmente obtener la predicción y generar el fichero para kaggle. 

In [72]:
# Construcción del pipeline final para Kaggle con el mejor preprocesador y modelo
# Construimos el preprocesador fijo con la mejor versión
preprocesor = make_preprocessor(dense_output=False, use_age_bins=False, use_surname=False)
best_model = models[best_model_name]
# Pipeline Completo (Preprocesamiento + Modelo)
best_model_pipeline = Pipeline(steps=[
    ('feature_engineer', feature_engineer),
    ('preprocessor', preprocesor),
    ('classifier', best_model)
])
# Configuramos y ejecutamos la Validación Cruzada local
cv_metrics = evaluate_pipeline(best_model_pipeline, X_train, y_train, n_splits=5)
# Generación de Submission para Kaggle con el mejor modelo encontrado
# Re-entrenamos con TODOS los datos de train para la predicción final
best_model_pipeline.fit(X_train, y_train) 
test_predictions = best_model_pipeline.predict(test_df)

# Crear fichero de salida
submission_df = pd.DataFrame({
    'CustomerId': test_df['CustomerId'],
    'Exited': test_predictions
})
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Fichero '{SUBMISSION_PATH}' generado correctamente.")

print("\n---- Mejores Resultados y Validación Cruzada local -----")
print("Mejor modelo:", best_model_name)
print("Mejor versión de preprocesador:", best_preprocesor_version)
print(f"Mean F1-Score:  {cv_metrics['f1_mean']:.4f} (+/- Std {cv_metrics['f1_std']:.4f})")
print(f"Mean Accuracy:  {cv_metrics['accuracy_mean']:.4f} (+/- Std {cv_metrics['accuracy_std']:.4f})")
print(f"Mean Kappa:     {cv_metrics['kappa_mean']:.4f}")
print(f"Mean Precision: {cv_metrics['precision_mean']:.4f}")
print(f"Mean Recall:    {cv_metrics['recall_mean']:.4f}")



TypeError: make_preprocessor() got an unexpected keyword argument 'dense_output'